# Fine-tune a larger open model (QLoRA) and score it (Colab)
Same data and settings as run I, on a bigger base model. Edit the **settings cell**, then run all cells.

Upload `fallacy_code.zip` and `external_eval.zip` (private, local use only), and optionally `distill_real.zip` (the LLM-labelled real-style data from option B, **private, local use only**). Needs the `HF_TOKEN` secret with *Notebook access* on.

**Speed (rough guess, please check the first progress lines and stop if it is too slow):**
- Run I (1.5B, fp16 on a T4) took 1.27 s/step at batch 8, 5 epochs, 77 minutes.
- A 7-9B model in 4-bit with gradient checkpointing is several times slower per step. On a T4 that probably means many hours, so use **an L4 or A100** (Colab Pro / pay-as-you-go compute units) and 2-3 epochs.
- The notebook saves the adapter after every epoch (`out/lora_X`), and it can be downloaded early.

In [ ]:
# ---- settings ----
BASE = "Qwen/Qwen2.5-7B-Instruct"      # alternatives: "unsloth/gemma-2-9b-it" (needs bf16: L4/A100), "unsloth/Llama-3.1-8B-Instruct"
RUN = "K"                               # name used for the output files
EPOCHS = 3
BATCH = 4                               # 4 fits a 16 GB GPU with 4-bit loading; raise to 8 on an A100
LR = 1e-4                               # lower than the 1.5B runs (2e-4) because the model is bigger
RANK = 16

In [ ]:
!nvidia-smi
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm huggingface_hub bitsandbytes sentencepiece

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less memory fragmentation
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()  # the secret can only be read from the notebook process
!ls -la /content/*.zip
!rm -rf /content/work && mkdir -p /content/work
!for z in /content/fallacy_code.zip /content/external_eval.zip; do unzip -tq $z || echo "BROKEN (incomplete upload): $z"; unzip -oq $z -d /content/work; done
%cd /content/work
!python fetch_data.py --rev v5
!wc -l data/generated/*.jsonl external/*.jsonl

In [ ]:
!for z in /content/distill_real.zip; do [ -f $z ] && unzip -oq $z -d /content/work && echo "distilled data added"; done
!wc -l data/generated/distill_real.jsonl 2>/dev/null || echo "no distilled data (fine)"

## Timing probe (20 steps, about 1-3 minutes): look at s/step in the progress bar, then decide whether this GPU is fast enough
Estimated total time = s/step x (training examples / batch) x EPOCHS. Example: 6 s/step x (5,822 / 4) x 3 = about 7 hours (too slow on this GPU).

In [ ]:
!python train.py --base {BASE} --four-bit --out out/probe --epochs 1 --batch {BATCH} --lr {LR} --rank {RANK} --val-per-label 0 --max-steps 20

## Train

In [ ]:
!python train.py --base {BASE} --four-bit --out out/lora_{RUN} --epochs {EPOCHS} --batch {BATCH} --lr {LR} --rank {RANK} --val-per-label 0 2>&1 | tee train_{RUN}.log

## Quick evaluation (strict accuracy and false-alarm rate)

In [ ]:
!python evaluate.py --base {BASE} --four-bit --adapter out/lora_{RUN} --out lora_{RUN}_preds.jsonl
!python evaluate.py --base {BASE} --four-bit --adapter out/lora_{RUN} --eval-file data/generated/eval_none.jsonl --out none_{RUN}_preds.jsonl
!python evaluate.py --base {BASE} --four-bit --adapter out/lora_{RUN} --eval-file data/generated/eval_real.jsonl --out real_{RUN}_preds.jsonl

## Score every name on every set (for calibration and the real-style comparison)

In [ ]:
!python score_candidates.py --base {BASE} --four-bit --adapter out/lora_{RUN} --run {RUN} \
  --set eval=data/generated/eval.jsonl \
  --set eval_none=data/generated/eval_none.jsonl \
  --set eval_real=data/generated/eval_real.jsonl \
  --set calib_fallacy=data/generated/calib_fallacy.jsonl \
  --set calib_none=data/generated/calib_none.jsonl \
  --set cocolofa_calib=external/cocolofa_calib.jsonl \
  --set cocolofa_test=external/cocolofa_test.jsonl \
  --set mafalda_test=external/mafalda_test.jsonl \
  --set cocolofa_tradition_calib=external/cocolofa_tradition_calib.jsonl \
  --set cocolofa_tradition_test=external/cocolofa_tradition_test.jsonl

In [ ]:
!zip -qr result_{RUN}.zip out/lora_{RUN} data/scores_{RUN}_*.json data/lora_{RUN}_preds.jsonl data/none_{RUN}_preds.jsonl data/real_{RUN}_preds.jsonl train_{RUN}.log -x '*/README.md'
from google.colab import files
files.download(f'/content/work/result_{RUN}.zip')